# 02 - Data Cleaning (Bronze -> Silver)
Cleans, types, and enriches all 9 Bronze tables and saves 8 Delta tables to the Silver layer.

## Cell 1 - Read Bronze files

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customers   = spark.read.option("header", True).csv("Files/Bronze/customers.csv")
orders      = spark.read.option("header", True).csv("Files/Bronze/orders.csv")
items       = spark.read.option("header", True).csv("Files/Bronze/order_items.csv")
products    = spark.read.option("header", True).csv("Files/Bronze/products.csv")
payments    = spark.read.option("header", True).csv("Files/Bronze/order_payments_dataset.csv")
reviews     = spark.read.option("header", True).csv("Files/Bronze/order_reviews.csv")
sellers     = spark.read.option("header", True).csv("Files/Bronze/sellers.csv")
geo         = spark.read.option("header", True).csv("Files/Bronze/geolocation.csv")
translation = spark.read.option("header", True).csv("Files/Bronze/product_category_name_translation.csv")

## Cell 2 - Clean customers

In [ ]:
customers = (customers
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("integer"))
    .withColumn("customer_city", initcap(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .dropDuplicates(["customer_id"]))

customers.show(5)

## Cell 3 - Clean orders (date parsing + feature engineering)

In [ ]:
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for c in date_cols:
    orders = orders.withColumn(c, to_timestamp(col(c), "M/d/yyyy H:mm"))

orders = (orders
    .withColumn("OrderYear", year("order_purchase_timestamp"))
    .withColumn("OrderMonth", month("order_purchase_timestamp"))
    .withColumn("OrderDay", dayofmonth("order_purchase_timestamp"))
    .withColumn("MonthName", date_format("order_purchase_timestamp", "MMMM"))
    .withColumn("Quarter", quarter("order_purchase_timestamp"))
    .withColumn("WeekDay", date_format("order_purchase_timestamp", "EEEE"))
    .withColumn("DeliveryDays", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("LateDeliveryDays", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
    .withColumn("DeliveryStatus",
        when(col("order_delivered_customer_date").isNull(), "Not Delivered Yet")
        .when(col("LateDeliveryDays") > 0, "Late")
        .otherwise("On Time"))
    .dropDuplicates(["order_id"]))

orders.show(5)

## Cell 4 - Clean order_items

In [ ]:
items = (items
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("order_item_id", col("order_item_id").cast("integer"))
    .withColumn("TotalValue", col("price") + col("freight_value"))
    .dropDuplicates(["order_id", "order_item_id"]))

items.show(5)

## Cell 5 - Clean products (merge with category translation)

In [ ]:
products = (products
    .fillna({"product_category_name": "unknown"})
    .fillna(0, subset=["product_name_lenght", "product_description_lenght", "product_photos_qty",
                        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])
    .join(translation, "product_category_name", "left")
    .withColumn("product_category_name_english", coalesce(col("product_category_name_english"), lit("Unknown")))
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .dropDuplicates(["product_id"]))

products.show(5)

## Cell 6 - Clean payments

In [ ]:
payments = (payments
    .withColumn("payment_sequential", col("payment_sequential").cast("integer"))
    .withColumn("payment_installments", col("payment_installments").cast("integer"))
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("InstallmentBucket",
        when(col("payment_installments") <= 1, "Single Payment")
        .when(col("payment_installments") <= 6, "2-6 Installments")
        .otherwise("7+ Installments")))

payments.show(5)

## Cell 7 - Clean reviews

In [ ]:
reviews = (reviews
    .withColumn("review_score", col("review_score").cast("integer"))
    .fillna({"review_comment_title": "No Title", "review_comment_message": "No Comment"})
    .withColumn("HasWrittenReview", when(col("review_comment_message") != "No Comment", True).otherwise(False))
    .dropDuplicates(["review_id"]))

reviews.show(5)

## Cell 8 - Clean sellers

In [ ]:
sellers = (sellers
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("integer"))
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .dropDuplicates(["seller_id"]))

sellers.show(5)

## Cell 9 - Clean geolocation (remove ~261,836 duplicate rows + aggregate by zip prefix)

In [ ]:
geo_clean = (geo
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("integer"))
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
    .dropDuplicates())

geo_agg = (geo_clean.groupBy("geolocation_zip_code_prefix")
    .agg(avg("geolocation_lat").alias("lat"),
         avg("geolocation_lng").alias("lng"),
         first("geolocation_city").alias("city"),
         first("geolocation_state").alias("state")))

print("before:", geo.count(), "after dedup+agg:", geo_agg.count())
geo_agg.show(5)

## Cell 10 - Save all tables as Delta (Silver layer)

In [ ]:
silver_tables = {
    "silver_customers": customers,
    "silver_orders": orders,
    "silver_order_items": items,
    "silver_products": products,
    "silver_payments": payments,
    "silver_reviews": reviews,
    "silver_sellers": sellers,
    "silver_geolocation": geo_agg
}

for name, df in silver_tables.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(f"saved: {name}")